# 1. Runtime / GPU Check
Collect structured runtime facts. CPU rendering is intentionally forbidden.

In [ ]:
import json, platform, shutil, subprocess
import torch
cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
total_vram = torch.cuda.get_device_properties(0).total_memory if cuda_available else 0
free_vram = torch.cuda.mem_get_info(0)[0] if cuda_available else 0
RUNTIME_INFO = {
    'cuda_available': cuda_available, 'gpu_name': gpu_name,
    'total_vram_gb': round(total_vram / 2**30, 2), 'free_vram_gb': round(free_vram / 2**30, 2),
    'torch_version': torch.__version__, 'cuda_version': torch.version.cuda,
    'python_version': platform.python_version(), 'disk_free_gb': round(shutil.disk_usage('/content').free / 2**30, 2),
}
print(json.dumps(RUNTIME_INFO, indent=2))


# 2. Package Installation

In [ ]:
%pip install -q --upgrade 'diffusers==0.35.1' 'transformers>=4.44,<5' 'accelerate>=1.0,<2' 'safetensors>=0.4' 'fastapi>=0.115,<1' 'uvicorn[standard]>=0.30,<1' 'huggingface_hub>=0.27,<1'
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared


# 3. Model Configuration
All model and generation settings are centralized here.

In [ ]:
MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
MODEL_REVISION = 'main'
DTYPE = torch.float16
DEVICE = 'cuda'
LOW_VRAM = RUNTIME_INFO['total_vram_gb'] < 14
ENABLE_CPU_OFFLOAD = LOW_VRAM
ENABLE_SEQUENTIAL_CPU_OFFLOAD = LOW_VRAM
ENABLE_ATTENTION_SLICING = True
ENABLE_VAE_SLICING = True
ENABLE_XFORMERS = False
DEFAULT_WIDTH, DEFAULT_HEIGHT = ((768, 432) if LOW_VRAM else (1024, 576))
DEFAULT_STEPS = 30
DEFAULT_GUIDANCE = 6.5
print({'model': MODEL_ID, 'low_vram': LOW_VRAM, 'size': [DEFAULT_WIDTH, DEFAULT_HEIGHT], 'steps': DEFAULT_STEPS})


# 4. Model Download

In [ ]:
from huggingface_hub import snapshot_download
MODEL_PATH = None
MODEL_DOWNLOAD_ERROR = None
if cuda_available:
    try:
        MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REVISION)
    except Exception as exc:
        MODEL_DOWNLOAD_ERROR = f'{type(exc).__name__}: {exc}'
else:
    MODEL_DOWNLOAD_ERROR = 'CUDA GPU is unavailable; model download skipped.'
print({'model_path': MODEL_PATH, 'error': MODEL_DOWNLOAD_ERROR})


# 5. Pipeline Initialization

In [ ]:
from diffusers import StableDiffusionXLPipeline
pipe = None
PIPELINE_ERROR = MODEL_DOWNLOAD_ERROR
if cuda_available and MODEL_PATH:
    try:
        pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_PATH, torch_dtype=DTYPE, variant='fp16', use_safetensors=True, local_files_only=True)
        if ENABLE_ATTENTION_SLICING: pipe.enable_attention_slicing()
        if ENABLE_VAE_SLICING: pipe.enable_vae_slicing()
        if ENABLE_XFORMERS: pipe.enable_xformers_memory_efficient_attention()
        if ENABLE_SEQUENTIAL_CPU_OFFLOAD: pipe.enable_sequential_cpu_offload()
        elif ENABLE_CPU_OFFLOAD: pipe.enable_model_cpu_offload()
        else: pipe.to(DEVICE)
        PIPELINE_ERROR = None
    except Exception as exc:
        PIPELINE_ERROR = f'{type(exc).__name__}: {exc}'
        pipe = None
print({'model_loaded': pipe is not None, 'error': PIPELINE_ERROR})


# 6. Job Store
Terminal states are immutable and every job owns a unique output path.

In [ ]:
import secrets, threading, time, uuid
from datetime import datetime, timezone
from pathlib import Path
ALLOWED_STATES = {'queued','loading_model','sampling','decoding','saving','completed','failed','cancelled'}
TERMINAL_STATES = {'completed','failed','cancelled'}
JOBS, JOB_LOCK = {}, threading.RLock()
OUTPUT_DIR = Path('/content/nova_outputs'); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def utc_now(): return datetime.now(timezone.utc).isoformat()
def public_job(job):
    keys = ('job_id','task_id','status','progress','current_step','preview_url','result_url','error')
    return {key: job.get(key) for key in keys}
def transition(job_id, status, progress=None, **updates):
    if status not in ALLOWED_STATES: raise ValueError(f'Invalid status: {status}')
    with JOB_LOCK:
        job = JOBS[job_id]
        if job['status'] in TERMINAL_STATES and job['status'] != status: return False
        if job.get('cancelled') and status != 'cancelled': return False
        job.update(status=status, current_step=status, **updates)
        if progress is not None: job['progress'] = max(job['progress'], float(progress))
        return True
API_TOKEN = secrets.token_urlsafe(32)


# 7. Image Generation Worker

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
ARCHVIZ_PROMPT = 'photorealistic architectural visualization of a luxury commercial cafe interior, professional interior design presentation, coherent spatial planning, accurate furniture scale, warm natural wood, glass facade, stone and brushed metal details, physically based materials, realistic global illumination, soft cinematic lighting, high-end hospitality design, wide angle interior photography, clean composition, realistic perspective, high detail, commercially usable interior concept, no people'
DEFAULT_NEGATIVE = 'low quality, blurry, cartoon, illustration, low poly, distorted furniture, floating objects, impossible architecture, warped walls, duplicate furniture, bad perspective, fisheye distortion, overexposed, underexposed, text, logo, watermark, people, hands, faces'
class JobCancelled(Exception): pass
def render_worker(job_id):
    started = time.monotonic()
    try:
        if pipe is None or not cuda_available: raise RuntimeError('GPU_UNAVAILABLE')
        if not transition(job_id, 'loading_model', .02, started_at=utc_now()): return
        job = JOBS[job_id]
        transition(job_id, 'sampling', .05)
        generator = torch.Generator(device='cuda').manual_seed(job['seed'])
        def on_step(_pipeline, step, _timestep, callback_kwargs):
            if JOBS[job_id].get('cancelled'): raise JobCancelled()
            if step + 1 >= job['steps']: transition(job_id, 'decoding', .90)
            else: transition(job_id, 'sampling', .05 + .80 * ((step + 1) / job['steps']))
            return callback_kwargs
        result = pipe(prompt=job['prompt'], negative_prompt=job['negative_prompt'], width=job['width'], height=job['height'], num_inference_steps=job['steps'], guidance_scale=job['guidance_scale'], generator=generator, callback_on_step_end=on_step, output_type='pil')
        if JOBS[job_id]['status'] != 'decoding' and not transition(job_id, 'decoding', .90): return
        image = result.images[0]
        if not transition(job_id, 'saving', .95): return
        path = OUTPUT_DIR / f"{job['task_id']}_{job_id}_{job['seed']}.png"
        image.save(path, format='PNG')
        with Image.open(path) as check:
            check.load(); width, height = check.size
        transition(job_id, 'completed', 1.0, result_path=str(path), result_url=f'/result/{job_id}', actual_width=width, actual_height=height, completed_at=utc_now(), generation_seconds=round(time.monotonic()-started, 2))
    except JobCancelled:
        transition(job_id, 'cancelled', JOBS[job_id]['progress'], cancelled=True, completed_at=utc_now())
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        transition(job_id, 'failed', JOBS[job_id]['progress'], error={'code':'GPU_OUT_OF_MEMORY','message':str(exc),'retryable':True}, completed_at=utc_now())
    except Exception as exc:
        code = 'GPU_UNAVAILABLE' if str(exc) == 'GPU_UNAVAILABLE' else 'RENDER_FAILED'
        transition(job_id, 'failed', JOBS[job_id]['progress'], error={'code':code,'message':str(exc),'retryable':code!='GPU_UNAVAILABLE'}, completed_at=utc_now())
EXECUTOR = ThreadPoolExecutor(max_workers=1, thread_name_prefix='nova-render')


# 8. FastAPI App

In [ ]:
from fastapi import Depends, FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
app = FastAPI(title='NOVA Colab Render Backend', version='2B')
class RenderBody(BaseModel):
    task_id: str = Field(min_length=1, max_length=128)
    prompt: str = Field(min_length=1, max_length=6000)
    negative_prompt: str = ''
    width: int = DEFAULT_WIDTH; height: int = DEFAULT_HEIGHT; steps: int = DEFAULT_STEPS
    guidance_scale: float = DEFAULT_GUIDANCE; seed: int = -1
    style: str = 'photorealistic_archviz'; project_type: str = 'luxury_cafe_interior'; metadata: dict = {}
@app.get('/health')
def health():
    ready = bool(cuda_available and pipe is not None)
    return {'provider':'google_colab','available':ready,'gpu_available':cuda_available,'model_loaded':pipe is not None,'gpu_name':gpu_name,'detail':'ready' if ready else (PIPELINE_ERROR or 'GPU unavailable'),'checked_at':utc_now()}
@app.post('/render', status_code=202)
def submit(body: RenderBody):
    if not cuda_available or pipe is None: raise HTTPException(503, detail={'code':'GPU_UNAVAILABLE','message':PIPELINE_ERROR or 'CUDA GPU unavailable','retryable':False})
    width, height = min(body.width, DEFAULT_WIDTH), min(body.height, DEFAULT_HEIGHT)
    width, height = width - width % 8, height - height % 8
    job_id = uuid.uuid4().hex; seed = body.seed if body.seed >= 0 else secrets.randbelow(2**31)
    job = {'job_id':job_id,'task_id':body.task_id,'status':'queued','progress':0.0,'current_step':'queued','prompt':f'{body.prompt}, {ARCHVIZ_PROMPT}','negative_prompt':f'{body.negative_prompt}, {DEFAULT_NEGATIVE}'.strip(', '),'width':width,'height':height,'steps':min(max(body.steps,25),35),'guidance_scale':body.guidance_scale,'seed':seed,'result_path':None,'result_url':None,'preview_url':None,'error':None,'created_at':utc_now(),'started_at':None,'completed_at':None,'cancelled':False,'metadata':body.metadata}
    with JOB_LOCK: JOBS[job_id] = job
    EXECUTOR.submit(render_worker, job_id)
    return {key:job[key] for key in ('job_id','task_id','status','created_at')}
@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOB_LOCK:
        if job_id not in JOBS: raise HTTPException(404, 'job not found')
        return public_job(JOBS[job_id])
@app.get('/jobs/{job_id}/result')
def result_metadata(job_id: str):
    with JOB_LOCK:
        job = JOBS.get(job_id)
        if not job: raise HTTPException(404, 'job not found')
        if job['status'] != 'completed': raise HTTPException(409, 'job not completed')
        return {'job_id':job_id,'task_id':job['task_id'],'status':'completed','image_url':job['result_url'],'width':job['actual_width'],'height':job['actual_height'],'seed':job['seed'],'metadata':{'model':MODEL_ID,'generation_seconds':job['generation_seconds']}}
@app.get('/result/{job_id}')
def result_image(job_id: str):
    with JOB_LOCK:
        job = JOBS.get(job_id)
        if not job: raise HTTPException(404, 'job not found')
        if job['status'] != 'completed': raise HTTPException(409, 'job not completed')
        path = job['result_path']
    return FileResponse(path, media_type='image/png', filename=Path(path).name)
def cancel_impl(job_id):
    with JOB_LOCK:
        job = JOBS.get(job_id)
        if not job: raise HTTPException(404, 'job not found')
        if job['status'] in TERMINAL_STATES: return public_job(job)
        job['cancelled'] = True
    transition(job_id, 'cancelled', job['progress'], cancelled=True, completed_at=utc_now())
    return public_job(JOBS[job_id])
@app.post('/jobs/{job_id}/cancel')
def cancel_post(job_id: str): return cancel_impl(job_id)
@app.delete('/jobs/{job_id}')
def cancel_delete(job_id: str): return cancel_impl(job_id)


# 9. Authentication
Replace placeholder dependencies with constant-time Bearer validation. The token is never logged.

In [ ]:
from fastapi import Request
@app.middleware('http')
async def bearer_auth(request: Request, call_next):
    scheme, _, value = request.headers.get('Authorization', '').partition(' ')
    if scheme.lower() != 'bearer' or not secrets.compare_digest(value, API_TOKEN):
        from fastapi.responses import JSONResponse
        return JSONResponse(status_code=401, content={'detail':'unauthorized'})
    return await call_next(request)
print('Bearer authentication enabled; token fingerprint:', API_TOKEN[-4:].rjust(len(API_TOKEN), '*'))


# 10. Cloudflare Tunnel

In [ ]:
import re, requests
import uvicorn
SERVER = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning', access_log=False))
SERVER_THREAD = threading.Thread(target=SERVER.run, daemon=True); SERVER_THREAD.start()
for _ in range(50):
    try:
        requests.get('http://127.0.0.1:8000/health', headers={'Authorization':f'Bearer {API_TOKEN}'}, timeout=1); break
    except requests.RequestException: time.sleep(.2)
TUNNEL = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
TUNNEL_URL = None
deadline = time.time() + 30
while time.time() < deadline:
    line = TUNNEL.stdout.readline()
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match: TUNNEL_URL = match.group(0); break
if not TUNNEL_URL: raise RuntimeError('Cloudflare tunnel URL was not created')
print('Tunnel:', TUNNEL_URL)


# 11. Health Test

In [ ]:
AUTH = {'Authorization': f'Bearer {API_TOKEN}'}
health_response = requests.get(f'{TUNNEL_URL}/health', headers=AUTH, timeout=30)
health_response.raise_for_status()
print(json.dumps(health_response.json(), indent=2))


# 12. Render Test
This submits a new GPU job and downloads only that job's result.

In [ ]:
TEST_TASK_ID = f'cafe-live-{uuid.uuid4().hex[:8]}'
payload = {'task_id':TEST_TASK_ID,'prompt':'Design a premium modern cafe interior with warm oak wood, a large glass facade, soft indirect lighting, stone service counter, comfortable seating, clear circulation, luxury commercial hospitality design, photorealistic architectural visualization.','negative_prompt':DEFAULT_NEGATIVE,'width':DEFAULT_WIDTH,'height':DEFAULT_HEIGHT,'steps':DEFAULT_STEPS,'guidance_scale':DEFAULT_GUIDANCE,'seed':secrets.randbelow(2**31),'style':'photorealistic_archviz','project_type':'luxury_cafe_interior','metadata':{'test':'phase2b-live'}}
submitted = requests.post(f'{TUNNEL_URL}/render', headers=AUTH, json=payload, timeout=60); submitted.raise_for_status()
LIVE_JOB = submitted.json(); print('submitted', LIVE_JOB)
seen = []
while True:
    status_response = requests.get(f"{TUNNEL_URL}/jobs/{LIVE_JOB['job_id']}", headers=AUTH, timeout=60); status_response.raise_for_status()
    current = status_response.json()
    if not seen or seen[-1] != current['status']: seen.append(current['status']); print(current)
    if current['status'] in TERMINAL_STATES: break
    time.sleep(2)
if current['status'] != 'completed': raise RuntimeError(current.get('error'))
image_response = requests.get(f"{TUNNEL_URL}/result/{LIVE_JOB['job_id']}", headers=AUTH, timeout=120); image_response.raise_for_status()
if not image_response.headers.get('Content-Type','').startswith('image/'): raise RuntimeError('Result was not an image')
LIVE_IMAGE_PATH = Path('/content') / f"{TEST_TASK_ID}_{LIVE_JOB['job_id']}.png"; LIVE_IMAGE_PATH.write_bytes(image_response.content)
with Image.open(LIVE_IMAGE_PATH) as verified: verified.load(); print({'job_id':LIVE_JOB['job_id'],'states':seen,'size':verified.size,'path':str(LIVE_IMAGE_PATH)})
display(Image.open(LIVE_IMAGE_PATH))


# 13. Client Configuration Output
Copy manually into local `.env`. Do not commit these runtime values.

In [ ]:
print(f'NOVA_COLAB_BASE_URL={TUNNEL_URL}')
print(f'NOVA_COLAB_TOKEN={API_TOKEN}')
print('NOVA_COLAB_CONNECT_TIMEOUT_SECONDS=10')
print('NOVA_COLAB_READ_TIMEOUT_SECONDS=60')
print('NOVA_COLAB_POLL_INTERVAL_SECONDS=2')
print('NOVA_COLAB_MAX_POLL_SECONDS=900')
print('NOVA_COLAB_MAX_RESULT_BYTES=26214400')


# 14. Shutdown / Cleanup

In [ ]:
if 'TUNNEL' in globals() and TUNNEL.poll() is None: TUNNEL.terminate()
if 'SERVER' in globals(): SERVER.should_exit = True
if 'EXECUTOR' in globals(): EXECUTOR.shutdown(wait=False, cancel_futures=True)
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('NOVA Colab backend stopped.')
